# Week 1 Exercise — Technical question explainer

**Assignment:** Build a tool that takes a technical question and responds with an explanation, using both a cloud API (OpenAI/OpenRouter) and Ollama.

This solution uses **OpenRouter** for the cloud model and **Ollama** (local) for the second model. Set `OPENROUTER_API_KEY` in `.env`. For Ollama, run `ollama pull llama3.2` and have Ollama running.

In [ ]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants
MODEL_CLOUD = "openai/gpt-4o-mini"   # via OpenRouter
MODEL_OLLAMA = "llama3.2"             # local; run: ollama pull llama3.2

In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("Add OPENROUTER_API_KEY to .env")
else:
    print("OpenRouter API key loaded.")

client_cloud = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
client_ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
# type your question here
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
system_prompt = "You are a helpful technical tutor who answers questions about Python, software engineering, data science, and LLMs."
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Please give a clear explanation: " + question.strip()}
]

## Answer from OpenRouter (gpt-4o-mini) with streaming

In [ ]:
stream = client_cloud.chat.completions.create(
    model=MODEL_CLOUD,
    messages=messages,
    stream=True
)
response_text = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        response_text += chunk.choices[0].delta.content
        update_display(Markdown(response_text), display_id=display_handle.display_id)

## Answer from Ollama (llama3.2, local)

In [ ]:
response_ollama = client_ollama.chat.completions.create(
    model=MODEL_OLLAMA,
    messages=messages,
)
display(Markdown(response_ollama.choices[0].message.content))